# Scraping Data Penyakit dari Plantix

Notebook ini mengambil data melalui **Browse by Crop**: buka halaman library → buka crop yang diinginkan lalu kumpulkan URL detail → ekstraksi setiap halaman → simpan JSON dan CSV.

## Step 1 — Import Library

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

## Step 2 — Konfigurasi dan Inisialisasi WebDriver

Memakai Selenium Manager untuk mengelola ChromeDriver secara otomatis.

In [2]:
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(options=options)

## Step 3 — Pilih Tanaman dan Buka Halaman Browse by Crop

![Gambar](crop.png)

Kode asli membuka halaman `rice`. Variabel berikut memudahkan penggantian crop tanpa mengubah kode pengumpulan tautan.

In [ ]:
crop = "rice"
crop_url = f"https://plantix.net/en/library/crops/{crop}/"

#ganti ke bagian "en" menjadi "id" jika ingin berbahasa indonesia crop_url = f"https://plantix.net/id/library/crops/{crop}/"

driver.get(crop_url)

## Step 4 — Kumpulkan dan Deduplikasi URL Detail

Tautan diambil dari daftar pada halaman crop dan dideduplikasi dengan `Set`

In [4]:
selector = (
    'section[data-cy="crop-disease-link-list"] '
    'a[href*="/plant-diseases/"]'
)

WebDriverWait(driver, 30).until(
    lambda d: d.execute_script(
        "return document.querySelectorAll(arguments[0]).length > 0;",
        selector
    )
)

links = driver.execute_script("""
    const anchors = document.querySelectorAll(arguments[0]);
    return [...new Set(Array.from(anchors, a => a.href))];
""", selector)

print("URL:", driver.current_url)
print("Judul:", driver.title)
print("Total link:", len(links))
print(links[:5])

URL: https://plantix.net/en/library/crops/rice/
Judul: Rice | Crops | Plantix
Total link: 86
['https://plantix.net/en/library/plant-diseases/100180/bakanae-and-foot-rot/', 'https://plantix.net/en/library/plant-diseases/100058/blast-of-rice/', 'https://plantix.net/en/library/plant-diseases/100064/brown-spot-of-rice/', 'https://plantix.net/en/library/plant-diseases/100162/false-smut/', 'https://plantix.net/en/library/plant-diseases/100132/foot-and-collar-rot/']


## Step 5 — Definisikan Fungsi Ekstraksi Detail

In [5]:

def scrape_detail(url):
    driver.get(url)
    time.sleep(2)

    try:
        nama = driver.find_element(By.CLASS_NAME, "disease-name").text.strip()
    except Exception:
        try:
            nama = driver.find_element(By.TAG_NAME, "h1").text.strip()
        except Exception:
            nama = ""

    try:
        nama_ilmiah = driver.find_element(By.CLASS_NAME, "scientific-name").text.strip()
    except Exception:
        try:
            nama_ilmiah = driver.find_element(By.CSS_SELECTOR, "h1 + h2").text.strip()
        except Exception:
            nama_ilmiah = ""

    try:
        jenis = driver.find_element(By.CLASS_NAME, "class_text").text.strip()
    except Exception:
        try:
            jenis = driver.find_element(By.CSS_SELECTOR, "p.disease-type").text.strip()
        except Exception:
            jenis = ""

    try:
        gejala = driver.find_element(By.CSS_SELECTOR, '[data-cy="symptoms-card"] p').text.strip()
    except Exception:
        gejala = ""

    # Pengendalian biologis
    try:
        orgs = driver.find_elements(
            By.CSS_SELECTOR, '[data-cy="biological-control-card"] p'
        )
        pengendalian_biologis = " ".join(
            o.text.strip() for o in orgs if o.text.strip()
        )
    except Exception:
        pengendalian_biologis = ""

    # Pengendalian kimia
    try:
        chems = driver.find_elements(
            By.CSS_SELECTOR, '[data-cy="chemical-control-card"] p'
        )
        pengendalian_kimia = " ".join(
            c.text.strip() for c in chems if c.text.strip()
        )
    except Exception:
        pengendalian_kimia = ""

    try:
        penyebab = driver.find_element(
            By.CSS_SELECTOR, '[data-cy="trigger-card"] p'
        ).text.strip()
    except Exception:
        penyebab = ""

    try:
        lis = driver.find_elements(
            By.CSS_SELECTOR, '[data-cy="preventive-measures-card"] li'
        )
        pencegahan = "; ".join(
            li.text.strip() for li in lis if li.text.strip()
        )
    except Exception:
        pencegahan = ""

    return {
        "nama": nama,
        "namaIlmiah": nama_ilmiah,
        "jenis": jenis,
        "gejala": gejala,
        "pengendalianBiologis": pengendalian_biologis,
        "pengendalianKimia": pengendalian_kimia,
        "penyebab": penyebab,
        "pencegahan": pencegahan
    }


## Step 6 — Ekstraksi Data dari Seluruh URL

Semua URL dari Step 4 diproses satu per satu.

Jika perlu mengetes selector secara manual sebelum menjalankan Step 6, buat cell sementara berisi `scrape_detail(links[0])`.

In [6]:
hasil = []

for i, url in enumerate(links):
    print(f"[{i+1}/{len(links)}] Scraping: {url}")

    try:
        data = scrape_detail(url)
        hasil.append(data)
    except Exception as e:
        print("Error:", url, e)

    time.sleep(1)

[1/86] Scraping: https://plantix.net/en/library/plant-diseases/100180/bakanae-and-foot-rot/
[2/86] Scraping: https://plantix.net/en/library/plant-diseases/100058/blast-of-rice/
[3/86] Scraping: https://plantix.net/en/library/plant-diseases/100064/brown-spot-of-rice/
[4/86] Scraping: https://plantix.net/en/library/plant-diseases/100162/false-smut/
[5/86] Scraping: https://plantix.net/en/library/plant-diseases/100132/foot-and-collar-rot/
[6/86] Scraping: https://plantix.net/en/library/plant-diseases/100290/kernel-smut-of-rice/
[7/86] Scraping: https://plantix.net/en/library/plant-diseases/100182/leaf-scald-of-rice/
[8/86] Scraping: https://plantix.net/en/library/plant-diseases/100183/narrow-brown-leaf-spot-of-rice/
[9/86] Scraping: https://plantix.net/en/library/plant-diseases/100194/phoma-sorghina-in-rice/
[10/86] Scraping: https://plantix.net/en/library/plant-diseases/800054/red-stripe-disease/
[11/86] Scraping: https://plantix.net/en/library/plant-diseases/100080/rice-sheath-blight/
[

## Step 7 — Validasi Hasil Scraping

Memeriksa jumlah URL yang diproses, jumlah data yang diperoleh, dan jumlah nilai kosong per kolom.

In [7]:
df = pd.DataFrame(hasil)

print("Total URL ditemukan:", len(links))
print("Total data berhasil diekstraksi:", len(df))
print("Jumlah nilai kosong per kolom:")
print(df.isna().sum())

df.head()

Total URL ditemukan: 86
Total data berhasil diekstraksi: 86
Jumlah nilai kosong per kolom:
nama                    0
namaIlmiah              0
jenis                   0
gejala                  0
pengendalianBiologis    0
pengendalianKimia       0
penyebab                0
pencegahan              0
dtype: int64


,nama,namaIlmiah,jenis,gejala,pengendalianBiologis,pengendalianKimia,penyebab,pencegahan
0,Bakanae and Foot Rot,Gibberella fujikuroi,Fungus,Bakanae is conspicuous as a seedling disease b...,No biological treatment against this disease i...,Always consider an integrated approach with pr...,Bakanae is a seedborne fungal disease. The dis...,Use clean seeds to minimize the occurrence of ...
1,Blast of Rice,Magnaporthe oryzae,Fungus,Rice blast affects all the above-ground parts ...,"To this day, no effective biological control o...",Always consider an integrated approach with pr...,Symptoms of rice blast are caused by the fungu...,Use healthy or certified seed material.; Plant...
2,Brown Spot of Rice,Cochliobolus miyabeanus,Fungus,This disease is characterised by a wide range ...,To be sure that the seeds are not contaminated...,Always consider an integrated approach with bo...,"The symptoms are caused by the fungus, Cochlio...","For soils that are low in silicon, apply calci..."
3,False Smut,Villosiclava virens,Fungus,Symptoms are visible during the formation of t...,The treatment of seeds at 52° C for 10 min is ...,Always consider an integrated approach with pr...,The symptoms are caused by the fungus Villosic...,Use healthy seeds from certified retailers.; U...
4,Foot and Collar Rot,Athelia rolfsii,Fungus,"The fungus primarily attacks the stems, althou...",Antagonistic fungi (often in combination with ...,Always consider an integrated approach with pr...,The symptoms are caused by the fungus Athelia ...,Make sure to use healthy seeds from certified ...


## Step 8 — Simpan Hasil ke JSON dan CSV

Lokasi dan nama keluaran dipertahankan dari notebook asli: `data/raw/plantix.json` dan `data/raw/plantix.csv`.

In [8]:
output_dir = Path("data") / "raw"
output_dir.mkdir(parents=True, exist_ok=True)

with (output_dir / f"{crop}.json").open("w", encoding="utf-8") as f:
    json.dump(hasil, f, ensure_ascii=False, indent=2)

df.to_csv(output_dir / f"{crop}.csv", index=False)

print("JSON:", output_dir / f"{crop}.json")
print("CSV :", output_dir / f"{crop}.csv")

JSON: data\raw\rice.json
CSV : data\raw\rice.csv


## Step 9 — Tutup WebDriver

Jalankan setelah seluruh data selesai disimpan.

In [9]:
driver.quit()